In [ ]:
import csv
import json
import random
import re
from pathlib import Path
from typing import Dict, List, Tuple
import os
from pathlib import Path

# import your inflection helpers
from grammar import apply_tags

# ---------- basic Levenshtein (edit distance) ----------
def _levenshtein(a: str, b: str) -> int:
    a, b = a.lower(), b.lower()
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cost = 0 if ca == cb else 1
            cur.append(min(
                prev[j] + 1,      # deletion
                cur[j-1] + 1,     # insertion
                prev[j-1] + cost  # substitution
            ))
        prev = cur
    return prev[-1]

def _too_close(x: str, y: str, min_distance: int = 2) -> bool:
    # “less than 2” means distance 0 or 1 is too close
    return _levenshtein(x, y) < min_distance

# ---------- parsing placeholders ----------
# Matches: [noun_1][plural][capitalize], [verb_2][past], [rel_1] etc.
PH_RE = re.compile(
    r"(?P<prefix>(?:\[negate\])*)"                               # optional [negate]
    r"\[(?P<kind>noun|verb|adj|rel)_(?P<idx>\d+)\]"              # slot
    r"(?P<suffix>(?:\[(?:capitalize|plural|past|progressive|negate|abbr)\])*)"  # tags
)


def _extract_placeholders(template: str):
    """
    Return a list of (kind:str, idx:int, tags:List[str]) for each placeholder occurrence.
    Tags may include: negate, capitalize, plural, past, progressive.
    """
    results = []
    for m in PH_RE.finditer(template):
        kind = m.group("kind")
        idx = int(m.group("idx"))

        # collect tags from both prefix and suffix
        tags = []
        if m.group("prefix"):
            tags += re.findall(r"\[(negate)\]", m.group("prefix"))
        if m.group("suffix"):
            tags += re.findall(r"\[(capitalize|plural|past|progressive|negate|abbr)\]", m.group("suffix"))

        results.append((kind, idx, tags))
    return results

def _indices_by_kind(template: str):
    """
    Collect unique indices required per kind.
    """
    placeholders = _extract_placeholders(template)
    by_kind = {"noun": set(), "verb": set(), "adj": set(), "rel": set()}
    for kind, idx, _ in placeholders:
        by_kind[kind].add(idx)
    return {k: sorted(v) for k, v in by_kind.items()}

# ---------- selection with uniqueness + distance constraints ----------
def _choose_unique(words: List[str], disallow: List[str]) -> str:
    """
    Choose a word not equal to any in disallow and with edit distance >= 2 to each.
    """
    candidates = words[:]
    random.shuffle(candidates)
    for w in candidates:
        if all((w.lower() != d.lower()) and (not _too_close(w, d)) for d in disallow):
            return w
    raise ValueError("No available candidate satisfies uniqueness/distance constraints.")

def _build_assignment(template: str,
                      nouns: List[str],
                      verbs: List[str],
                      adjs: List[str],
                      rels: List[str]) -> Dict[str, Dict[int, str]]:
    """
    Returns mapping like {'noun': {1:'foo',2:'bar'}, 'verb': {1:'...'}, ...}
    ensuring no reuse within a prompt and edit distance >= 2 across all chosen items.
    """
    needed = _indices_by_kind(template)
    chosen: Dict[str, Dict[int, str]] = {"noun": {}, "verb": {}, "adj": {}, "rel": {}}
    used: List[str] = []

    pools = {
        "noun": nouns[:],
        "verb": verbs[:],
        "adj":  adjs[:],
        "rel":  rels[:],
    }

    # ensure we don't pick the same surface form across lists
    # (we also rely on distance constraint below)
    for kind in ("noun", "verb", "adj", "rel"):
        random.shuffle(pools[kind])

    for kind in ("noun", "verb", "adj", "rel"):
        for idx in needed[kind]:
            choice = _choose_unique(pools[kind], used)
            chosen[kind][idx] = choice
            used.append(choice)

    return chosen

# ---------- filling ----------
def _replacer_factory(assignments):
    """
    Build a replacement function for re.sub that applies tags with grammar.apply_tags.
    """
    def repl(m: re.Match) -> str:
        kind = m.group("kind")
        idx = int(m.group("idx"))

        # Gather tags from both sides exactly as in _extract_placeholders
        tags = []
        if m.group("prefix"):
            tags += re.findall(r"\[(negate)\]", m.group("prefix"))
        if m.group("suffix"):
            tags += re.findall(r"\[(capitalize|plural|past|progressive|negate|abbr)\]", m.group("suffix"))

        base = assignments[kind][idx]

        # Relations: ignore everything except 'capitalize'
        if kind == "rel":
            eff_tags = [t for t in tags if t == "capitalize"]
        else:
            eff_tags = tags

        return apply_tags(base, eff_tags)
    return repl

# ---------- main driver ----------
def create_prompts(template_path: str,
                   noun_path: str,
                   verb_path: str,
                   adj_path: str,
                   rel_path: str,
                   out_csv_path: str,
                   n_samples: int = 20,
                   seed: int = None) -> None:
    """
    Generate n_samples filled prompts with randomized, mutually distinct words
    (and edit distance >=2 between any two chosen items per prompt), then save to CSV.

    CSV columns:
      id, mapping (JSON), prompt
    """
    if seed is not None:
        random.seed(seed)

    template = Path(template_path).read_text(encoding="utf-8")

    nouns = [w.strip() for w in Path(noun_path).read_text(encoding="utf-8").splitlines() if w.strip()]
    verbs = [w.strip() for w in Path(verb_path).read_text(encoding="utf-8").splitlines() if w.strip()]
    adjs  = [w.strip() for w in Path(adj_path).read_text(encoding="utf-8").splitlines() if w.strip()]
    rels  = [w.strip() for w in Path(rel_path).read_text(encoding="utf-8").splitlines() if w.strip()]

    # sanity check: enough unique items per category
    needs = _indices_by_kind(template)
    for k, pool in (("noun", nouns), ("verb", verbs), ("adj", adjs), ("rel", rels)):
        if len(set(pool)) < len(needs[k]):
            raise ValueError(f"Not enough unique {k}s: need {len(needs[k])}, have {len(set(pool))}.")

    rows = []
    seen_prompts = set()

    for i in range(n_samples):
        # Retry a few times to avoid duplicates due to randomness/collisions
        for attempt in range(50):
            assignments = _build_assignment(template, nouns, verbs, adjs, rels)
            filled = PH_RE.sub(_replacer_factory(assignments), template).strip()

            if filled not in seen_prompts:
                seen_prompts.add(filled)
                mapping = {
                    "noun": assignments["noun"],
                    "verb": assignments["verb"],
                    "adj": assignments["adj"],
                    "rel": assignments["rel"],
                }
                rows.append({
                    "id": i + 1,
                    "mapping": json.dumps(mapping, ensure_ascii=False),
                    "prompt": filled
                })
                break
        else:
            # No unique prompt found after many attempts; stop early
            print(f"Stopped early at {len(rows)} prompts — reached uniqueness limit.")
            break

    # write CSV
    with open(out_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "mapping", "prompt"])
        writer.writeheader()
        writer.writerows(rows)


In [ ]:

create_prompts(
    template_path="../prompts/mystery_blocksworld_unfilled.txt",
    noun_path="../word_lists/nonce_noun.txt",
    verb_path="../word_lists/nonce_verb.txt",
    adj_path="../word_lists/nonce_adj.txt",
    rel_path="../word_lists/nonce_rel.txt",
    out_csv_path="../outputs/mystery_blocksworld_filled_prompts_500.csv",
    n_samples=500,
    seed=58
)


In [ ]:
# to run through all new versions

# Base directories to search
BASE_DIRS = [
    "../prompts/blocksworld/easy",
    "../prompts/blocksworld/medium",
    "../prompts/blocksworld/hard",
    "../prompts/blocksworld/super_hard",
    "../prompts/mystery_blocksworld/easy",
    "../prompts/mystery_blocksworld/medium",
    "../prompts/mystery_blocksworld/hard",
    "../prompts/mystery_blocksworld/super_hard"
]

# Word lists
NOUN_PATH = "../word_lists/nonce_noun.txt"
VERB_PATH = "../word_lists/nonce_verb.txt"
ADJ_PATH = "../word_lists/nonce_adj.txt"
REL_PATH = "../word_lists/nonce_rel.txt"

# Output base dirs, split by top-level category
OUT_BASE_BLOCKS = "../outputs/generated_prompts/blocksworld"
OUT_BASE_MYSTERY = "../outputs/generated_prompts/mystery_blocksworld"

# Number of samples per template
N_SAMPLES = 500
SEED = 58



for base_dir in BASE_DIRS:
    base_path = Path(base_dir)

    # Determine output root depending on which family we’re in
    if "mystery_blocksworld" in base_dir:
        out_root = Path(OUT_BASE_MYSTERY)
    else:
        out_root = Path(OUT_BASE_BLOCKS)

    # Find all .txt templates recursively in each directory
    for template_path in base_path.glob("*.txt"):
        # Build output subdir path to mirror prompts folder hierarchy after easy/medium/hard
        difficulty = template_path.parent.name
        out_dir = out_root / difficulty
        os.makedirs(out_dir, exist_ok=True)

        # Create output file name
        out_name = f"{template_path.stem}_{N_SAMPLES}.csv"
        out_path = out_dir / out_name

        print(f"Processing template: {template_path}")
        print(f" -> Output: {out_path}")

        # Generate prompts for this template
        create_prompts(
            template_path=str(template_path),
            noun_path=NOUN_PATH,
            verb_path=VERB_PATH,
            adj_path=ADJ_PATH,
            rel_path=REL_PATH,
            out_csv_path=str(out_path),
            n_samples=N_SAMPLES,
            seed=SEED
        )



Processing template: ../prompts/blocksworld/easy/blockworld_easy_actions.txt
 -> Output: ../outputs/generated_prompts/blocksworld/easy/blockworld_easy_actions_500.csv
Processing template: ../prompts/blocksworld/easy/blocksworld_easy_objects.txt
 -> Output: ../outputs/generated_prompts/blocksworld/easy/blocksworld_easy_objects_500.csv
Stopped early at 291 prompts — reached uniqueness limit.
Processing template: ../prompts/blocksworld/easy/blocksworld_easy_names.txt
 -> Output: ../outputs/generated_prompts/blocksworld/easy/blocksworld_easy_names_500.csv
Processing template: ../prompts/blocksworld/easy/blocksworld_easy_predicates.txt
 -> Output: ../outputs/generated_prompts/blocksworld/easy/blocksworld_easy_predicates_500.csv
Processing template: ../prompts/blocksworld/medium/blocksworld_medium_predicates_names.txt
 -> Output: ../outputs/generated_prompts/blocksworld/medium/blocksworld_medium_predicates_names_500.csv
Processing template: ../prompts/blocksworld/medium/blocksworld_medium_ac